# AI-ассистент для классификации ошибок (Math Error Classifier)

**Задача:** Multi-class classification (0-4).
**Метрика:** F1-macro.
**Модель:** Fine-tuning ruT5-base (безопасная версия кода).


In [ ]:
import pandas as pd
import numpy as np
import re
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from datasets import Dataset

# --- 1. Параметры ---
SEED = 42
MODEL_NAME = "ai-forever/ruT5-base"
MAX_LEN = 512
EPOCHS = 8
BATCH_SIZE = 4

np.random.seed(SEED)
torch.manual_seed(SEED)

# --- 2. Функции очистки (без сложных regex для json-безопасности) ---
def clean_text(text):
    if not isinstance(text, str):
        return ""
    
    # Упрощенная очистка LaTeX, чтобы не ломать токенизатор
    text = text.replace('\\frac', ' fraction ')
    text = text.replace('^{', ' power of ')
    text = text.replace('_', ' subscript ')
    text = text.replace('\\sqrt', ' square root ')
    text = text.replace('\\', '')
    text = text.replace('{', '').replace('}', '')
    text = text.replace(',', '.')
    return text.lower().strip()

def make_input(question, steps):
    q = question.lower().strip()
    s = clean_text(steps).strip()
    return "[QUESTION] " + q + " [SEP] " + s

# --- 3. Загрузка данных ---
print("Loading data...")
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

train_texts = []
for i in range(len(train_df)):
    train_texts.append(make_input(train_df.loc[i, 'question'], train_df.loc[i, 'steps']))

test_texts = []
for i in range(len(test_df)):
    test_texts.append(make_input(test_df.loc[i, 'question'], test_df.loc[i, 'steps']))

train_labels = train_df['error_category'].values

# --- 4. Токенизация ---
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=MAX_LEN)
test_encodings = tokenizer(test_texts, truncation=True, padding=True, max_length=MAX_LEN)

train_dataset = Dataset.from_dict({
    'input_ids': train_encodings['input_ids'],
    'attention_mask': train_encodings['attention_mask'],
    'label': train_labels
})

test_dataset = Dataset.from_dict({
    'input_ids': test_encodings['input_ids'],
    'attention_mask': test_encodings['attention_mask']
})

# --- 5. Обучение финальной модели ---
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=5)

args = TrainingArguments(
    output_dir="./output",
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    learning_rate=3e-5,
    warmup_steps=50,
    weight_decay=0.01,
    logging_strategy="epoch",
    save_strategy="no",
    seed=SEED,
    report_to="none",
    load_best_model_at_end=False
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
)

print("\nTraining final model on full data...")
trainer.train()

# --- 6. Предсказание и создание submission ---
preds = trainer.predict(test_dataset).predictions
labels = preds.argmax(axis=1)

submission = pd.DataFrame({
    'question_id': test_df['question_id'],
    'error_category': labels
})

submission.to_csv('submission.csv', index=False)
print("\nSUCCESS: submission.csv created.")
